Script: this piece of code runs the SEOF package, but over the time dimension on single file input for 500mb geopotential height in the PNA and NAt regions. This one has quadratic detrending not ensemble mean detrending!

Notes: needs to be run individually for each model - check the boxes that have a *change me* tag on the top. 

Output files: Standard EOFs for North Pacific & North Atlantic

PNA & NAT

SEOF, SPCS, FVAR, TVAR


In [2]:
#load packages needed in this notebook
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
from eofs.standard import Eof
import regionmask
import cartopy.crs as ccrs
from natsort import natsorted 

In [3]:
outputdir2='/glade/campaign/cgd/cas/nmaher/cesm2_lens/Amon/zg/' 
outputdir='/glade/work/nmaher/SEOF_output/'

model='CESM2'

#option for single files - put one file path here to get lon/lat/time
ds_fx = xr.open_dataset(outputdir2+'zg_mon_CESM2_cmip6_hist_ssp370_r1001.001i1p1f1_g025.nc')

lon = ds_fx.lon
lat = ds_fx.lat
time = ds_fx.time

In [4]:
#*change me* needs to be run for each model individually so for each i 

i=7

models=['CESM2','CESM1-LE','ACCESS-SSP370','CanESM5-SSP370','GFDL-SPEAR','IPSL-CM6A','MIROC6','MPI-GE','CESM2_cmip6']
model=models[i]
#tell me when the start year is 
syear=[1850,1920,1850,1850,1921,1850,1850,1850,1850]

offset=1950-syear[i]
    
eof_T='_zgDJF'
with np.load(outputdir+model+eof_T+'_SEOF.npz') as npz:
    data_z500 = np.ma.MaskedArray(**npz)
    

In [5]:
model

'MPI-GE'

In [6]:
#select 1950-2015
zg_DJF_full=data_z500[:,0+offset:66+offset,:,:]
zg_DJF_full.shape

(100, 66, 72, 144)

In [7]:
#remove the first time step as it is only JF not DJF
zg_DJF_2_full=zg_DJF_full[:,1:,:,:]

zg_DJF_2e_full = np.ma.masked_equal(np.zeros([len(zg_DJF_2_full[:,0,0,0]),65,72,144]),0)

ntt=len(zg_DJF_2e_full[:,0,0,0])

#DETREND HERE!
from statsmodels.tsa.tsatools import detrend
for k in range(ntt):
    detrended=np.zeros([65,72,144])
    for i in range(72):
        for j in range(144):
            detrended[:,i,j] = detrend(zg_DJF_2_full[k,:,i,j], order=2)
#remove the ensemble mean to get anomalies
    zg_DJF_2e_full[k,:,:,:] = detrended - np.ma.average(detrended,axis=0)




In [8]:
#mask the PNA region
lat_range=[20,90]
lon_range=[110,260]

lats=lat.values
lons=lon.values

ilat=np.logical_or(lats<20,lats>90)
ilon = np.logical_or(lons<110,lons>260)

ilat2d=np.broadcast_to(ilat[:,np.newaxis], zg_DJF_2e_full[0,0,...].shape)
ilon2d=np.broadcast_to(ilon[np.newaxis,:], zg_DJF_2e_full[0,0,...].shape)

mask2d=np.logical_or(ilat2d,ilon2d)
mask4d=np.broadcast_to(mask2d[np.newaxis,np.newaxis,:,:],zg_DJF_2e_full.shape)

masked_zgPNA=np.ma.masked_array(zg_DJF_2e_full,mask=mask4d)

In [9]:
#do the SEOF calculation

#weight with cosine of lat
coslat = np.cos(np.deg2rad(lat))

#how many eofs are we doing?
neof=3
ntt=len(zg_DJF_2_full[:,0,0,0])


#set up output dimensions
pna_seof = np.ma.masked_equal(np.zeros([ntt,neof,72,144]),0)
pna_spcs = np.ma.masked_equal(np.zeros([ntt,65,neof]),0)

pna_seof_fracVarExp_arr = np.ma.masked_equal(np.zeros([ntt,neof]),0)
pna_seof_totalVar_arr = np.ma.masked_equal(np.zeros([ntt,1]),0)
pna_eofs_northTest = np.ma.masked_equal(np.zeros([ntt,neof]),0)

for ii in range(ntt):
    masked_zgPNA2=np.squeeze(masked_zgPNA[ii,:,:,:])
    zg_DJF_3 = masked_zgPNA2.reshape(-1,72,144)

       
    a=np.sqrt(coslat)
    a2=a.values
    wgts = np.broadcast_to(a2[np.newaxis,:,np.newaxis], zg_DJF_3.shape)

    solver = Eof(zg_DJF_3, weights=wgts)


    eofs = solver.eofs(neofs=neof, eofscaling=2)
    pcs = solver.pcs(npcs=neof,pcscaling=1)
    fracvar = solver.varianceFraction(neigs=neof)
    total_variance = solver.totalAnomalyVariance()
    
    pna_eofs_northTest = solver.northTest(neigs=neof, vfscaled=True)
    
    spcs = np.ma.masked_array(np.zeros(pcs.shape),0)
    for j in range(len(pcs[0,:])):
        spcs[:,j] = (pcs[:,j] - np.mean(pcs[:,j]))/np.std(pcs[:,j])



    pna_seof[ii,...] = eofs
    pna_spcs[ii,...]= spcs
    pna_seof_fracVarExp_arr[ii,...]= fracvar
    pna_seof_totalVar_arr[ii,...] = total_variance


In [10]:
#save output
eof_T='_PNA'
outputdir_figD='/glade/work/nmaher/SEOF_output/Fig_data/'

np.savez_compressed(outputdir_figD+model+eof_T+'_SEOF_timequad.npz', data=pna_seof.data, mask=pna_seof.mask)
np.savez_compressed(outputdir_figD+model+eof_T+'_SPCS_timequad.npz', data=pna_spcs.data, mask=pna_spcs.mask)
np.savez_compressed(outputdir_figD+model+eof_T+'_FVAR_timequad.npz', data=pna_seof_fracVarExp_arr.data)
np.savez_compressed(outputdir_figD+model+eof_T+'_TVAR_timequad.npz', data=pna_seof_totalVar_arr.data)


In [11]:
#mask the NAt region
lat_range=[20,90]
lon_range=[280,10]

ilat=np.logical_or(lats<20,lats>90)
ilon = np.logical_and(lons>10,lons<280)

ilat2d=np.broadcast_to(ilat[:,np.newaxis], zg_DJF_2e_full[0,0,...].shape)
ilon2d=np.broadcast_to(ilon[np.newaxis,:], zg_DJF_2e_full[0,0,...].shape)

mask2d=np.logical_or(ilat2d,ilon2d)
mask4d=np.broadcast_to(mask2d[np.newaxis,np.newaxis,:,:],zg_DJF_2e_full.shape)

masked_zgNAT=np.ma.masked_array(zg_DJF_2e_full,mask=mask4d)

In [12]:
#do the SEOF calculation

#weight with cosine of lat
coslat = np.cos(np.deg2rad(lat))

#how many eofs are we doing?
neof=1

#set up output dimensions
nat_seof = np.ma.masked_equal(np.zeros([ntt,neof,72,144]),0)
nat_spcs = np.ma.masked_equal(np.zeros([ntt,65,neof]),0)

nat_seof_fracVarExp_arr = np.ma.masked_equal(np.zeros([ntt,neof]),0)
nat_seof_totalVar_arr = np.ma.masked_equal(np.zeros([ntt,1]),0)
nat_eofs_northTest = np.ma.masked_equal(np.zeros([ntt,neof]),0)


#loop through each year and do the EOFs
for ii in range(ntt):
    masked_zgNAT2=np.squeeze(masked_zgNAT[ii,:,:,:])
    zg_DJF_3 = masked_zgNAT2.reshape(-1,72,144)
    
   
    a=np.sqrt(coslat)
    a2=a.values
    wgts = np.broadcast_to(a2[np.newaxis,:,np.newaxis], zg_DJF_3.shape)

    solver = Eof(zg_DJF_3, weights=wgts)


    eofs = solver.eofs(neofs=neof, eofscaling=2)
    pcs = solver.pcs(npcs=neof,pcscaling=1)
    fracvar = solver.varianceFraction(neigs=neof)
    total_variance = solver.totalAnomalyVariance()
    
    nat_eofs_northTest = solver.northTest(neigs=neof, vfscaled=True)
    
    spcs = np.ma.masked_array(np.zeros(pcs.shape),0)
    for j in range(len(pcs[0,:])):
        spcs[:,j] = (pcs[:,j] - np.mean(pcs[:,j]))/np.std(pcs[:,j])


    nat_seof[ii,...]= eofs
    nat_spcs[ii,...]= spcs
    nat_seof_fracVarExp_arr[ii,...] = fracvar
    nat_seof_totalVar_arr[ii,...] = total_variance


In [13]:
#save output
eof_T='_NAT'

np.savez_compressed(outputdir_figD+model+eof_T+'_SEOF_timequad.npz', data=nat_seof.data, mask=nat_seof.mask)
np.savez_compressed(outputdir_figD+model+eof_T+'_SPCS_timequad.npz', data=nat_spcs.data, mask=nat_spcs.mask)
np.savez_compressed(outputdir_figD+model+eof_T+'_FVAR_timequad.npz', data=nat_seof_fracVarExp_arr.data)
np.savez_compressed(outputdir_figD+model+eof_T+'_TVAR_timequad.npz', data=nat_seof_totalVar_arr.data)